# Scripts para tabelas Delta dos CSVs

Este notebook contém scripts SQL `CREATE TABLE` para criar tabelas Delta no schema `workspace.bronze` a partir de 5 arquivos CSV localizados no volume `/Volumes/workspace/default/transacoes` (cartoes.csv, clientes.csv, contas.csv, estornos.csv, transacoes_cartao.csv). Cada célula utiliza a função `read_files` com inferência automática de schema. Os scripts estão prontos para serem executados.

In [0]:
%sql
-- ============================================================
-- Tabela: workspace.bronze.cartoes
-- Fonte:  /Volumes/workspace/default/transacoes/cartoes.csv
-- Descrição: Dados de cartões (cartão de crédito/débito) com
--            inferência automática de schema via read_files.
-- ============================================================
CREATE TABLE IF NOT EXISTS workspace.bronze.cartoes
USING DELTA
AS SELECT * FROM read_files(
  '/Volumes/workspace/default/transacoes/cartoes.csv',
  format => 'csv',
  header  => true
);

In [0]:
%sql
-- ============================================================
-- Tabela: workspace.bronze.clientes
-- Fonte:  /Volumes/workspace/default/transacoes/clientes.csv
-- Descrição: Dados de clientes com inferência automática de
--            schema via read_files.
-- ============================================================
CREATE TABLE IF NOT EXISTS workspace.bronze.clientes
USING DELTA
AS SELECT * FROM read_files(
  '/Volumes/workspace/default/transacoes/clientes.csv',
  format => 'csv',
  header  => true
);

In [0]:
%sql
-- ============================================================
-- Tabela: workspace.bronze.contas
-- Fonte:  /Volumes/workspace/default/transacoes/contas.csv
-- Descrição: Dados de contas bancárias com inferência automática
--            de schema via read_files.
-- ============================================================
CREATE TABLE IF NOT EXISTS workspace.bronze.contas
USING DELTA
AS SELECT * FROM read_files(
  '/Volumes/workspace/default/transacoes/contas.csv',
  format => 'csv',
  header  => true
);

In [0]:
%sql
-- ============================================================
-- Tabela: workspace.bronze.estornos
-- Fonte:  /Volumes/workspace/default/transacoes/estornos.csv
-- Descrição: Dados de estornos (reversões de transações) com
--            inferência automática de schema via read_files.
-- ============================================================
CREATE TABLE IF NOT EXISTS workspace.bronze.estornos
USING DELTA
AS SELECT * FROM read_files(
  '/Volumes/workspace/default/transacoes/estornos.csv',
  format => 'csv',
  header  => true
);

In [0]:
%sql
-- ============================================================
-- Tabela: workspace.bronze.transacoes_cartao
-- Fonte:  /Volumes/workspace/default/transacoes/transacoes_cartao_20250101.csv
-- Descrição: Dados de transações de cartão com inferência
--            automática de schema via read_files.
-- ============================================================
CREATE TABLE IF NOT EXISTS workspace.bronze.transacoes_cartao
USING DELTA
AS SELECT * FROM read_files(
  '/Volumes/workspace/default/transacoes/transacoes_cartao_20250101.csv',
  format => 'csv',
  header  => true
);

In [0]:
# Verificação do conteúdo dos arquivos CSV

import pandas as pd
import os

volume_path = "/Volumes/workspace/default/transacoes"
arquivos = [
    "cartoes.csv",
    "clientes.csv",
    "contas.csv",
    "estornos.csv",
    "transacoes_cartao_20250101.csv"
]

for arq in arquivos:
    caminho = os.path.join(volume_path, arq)
    if os.path.exists(caminho):
        df = pd.read_csv(caminho, nrows=5)
        print(f"\nArquivo: {arq} — Primeiros registros:")
        print(df)
        print(f"Total de linhas: {sum(1 for _ in open(caminho))-1}")
    else:
        print(f"Arquivo não encontrado: {arq}")

In [0]:
# Verificar se as tabelas Delta criadas estão com registros
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()
tabelas = [
    "workspace.bronze.cartoes",
    "workspace.bronze.clientes",
    "workspace.bronze.contas",
    "workspace.bronze.estornos",
    "workspace.bronze.transacoes_cartao"
]
for tab in tabelas:
    try:
        cnt = spark.table(tab).count()
        print(f"Tabela {tab}: {cnt} registros.")
    except Exception as e:
        print(f"Erro ao acessar {tab}: {e}")

## Camada Bronze: Pipeline seguindo governança atualizada

Fluxo automatizado de ingestão bruta e rastreável de múltiplos arquivos CSV (um ou mais por data de referência) para cada entidade, mantendo metadados técnicos essenciais.

**Etapas:**
- Ingerir todos os arquivos de um diretório por data
- Persistir *sem* transformações de negócio
- Adicionar colunas técnicas: arquivo_origem, data_referencia, timestamp_ingestao, hash_linha
- Pipeline seguro para reprocessamento idempotente e auditável.

O código abaixo exemplifica para todos os arquivos das entidades `cartoes`, `clientes`, `contas`, `estornos`, `transacoes_cartao` que terminem com `.csv` e estejam em `/Volumes/workspace/default/transacoes/`.

In [0]:
import os
from pyspark.sql.functions import input_file_name, lit, current_timestamp, sha1, concat_ws
from pyspark.sql.types import StringType
from datetime import datetime

# Parâmetros editáveis do pipeline
VOLUME_PATH = '/Volumes/workspace/default/transacoes/'
TABELAS = [
    ("cartoes", "cartoes.csv"),
    ("clientes", "clientes.csv"),
    ("contas", "contas.csv"),
    ("estornos", "estornos.csv"),
    ("transacoes_cartao", "transacoes_cartao_20250101.csv")
]
DATA_REFERENCIA = datetime.today().strftime('%Y-%m-%d') # ajuste se quiser passar como parâmetro
BRONZE_SCHEMA = 'workspace.bronze'

for entidade, arquivo in TABELAS:
    caminho = os.path.join(VOLUME_PATH, arquivo)
    if not os.path.exists(caminho):
        print(f"Arquivo não encontrado: {caminho}")
        continue
    df = spark.read.option("header", True).option("includeFileName", "true").csv(caminho)
    # Adiciona metadados técnicos usando _metadata.file_path no Unity Catalog
    import pyspark.sql.functions as F
    # Ajusta tipos para garantir compatibilidade de IDs
    for col_id in ['id_cartao', 'id_conta', 'id_cliente', 'id_transacao', 'id_estorno']:
        if col_id in df.columns:
            df = df.withColumn(col_id, F.col(col_id).cast('long'))
    df = df.withColumn("arquivo_origem", F.col("_metadata.file_path")) \
           .withColumn("data_referencia", lit(DATA_REFERENCIA)) \
           .withColumn("timestamp_ingestao", current_timestamp()) \
           .withColumn("hash_linha", sha1(concat_ws("||", *df.columns)))

    destino = f"{BRONZE_SCHEMA}.{entidade}"
    # Deduplicação por hash_linha — mantém apenas únicas por hash combinando ingestões
    from delta.tables import DeltaTable
    if spark.catalog.tableExists(destino):
        dt = DeltaTable.forName(spark, destino)
        # Junta dados novos e antigos, mantém apenas linhas únicas por hash_linha e timestamp_ingestao mais recente
        bronze_atual = dt.toDF()
        bronze_novo = df
        # Corrige diferença de colunas, especialmente _rescued_data
        todas_colunas = set(bronze_atual.columns) | set(bronze_novo.columns)
        bronze_atual_alinhado = bronze_atual.select([c if c in bronze_atual.columns else F.lit(None).alias(c) for c in todas_colunas])
        bronze_novo_alinhado = bronze_novo.select([c if c in bronze_novo.columns else F.lit(None).alias(c) for c in todas_colunas])
        union_df = bronze_atual_alinhado.unionByName(bronze_novo_alinhado)
        # Mantém o último registro de cada hash_linha
        from pyspark.sql.window import Window
        w = Window.partitionBy('hash_linha').orderBy(F.col('timestamp_ingestao').desc())
        bronze_dedup = union_df.withColumn('rn', F.row_number().over(w)).filter('rn = 1').drop('rn')
        bronze_dedup.write.format('delta').mode('overwrite').option('overwriteSchema', 'true').saveAsTable(destino)
        print(f"Reprocessamento OK (dedup): {arquivo} para {destino} ({bronze_dedup.count()} únicos)")
    else:
        df.write.format('delta').mode('overwrite').option('mergeSchema', 'true').saveAsTable(destino)
        print(f"Carga inicial OK: {arquivo} para {destino} ({df.count()} registros)")

In [0]:
# Recriação das tabelas bronze para garantir tipos corretos
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()
from pyspark.sql import DataFrame

tabelas = [
    ("cartoes", "cartoes.csv"),
    ("clientes", "clientes.csv"),
    ("contas", "contas.csv"),
    ("estornos", "estornos.csv"),
    ("transacoes_cartao", "transacoes_cartao_20250101.csv")
]
schema_bronze = "workspace.bronze"
volume_path = "/Volumes/workspace/default/transacoes/"

for entidade, arquivo in tabelas:
    destino = f"{schema_bronze}.{entidade}"
    # Remove tabela antiga se existir
    try:
        spark.sql(f"DROP TABLE IF EXISTS {destino}")
        print(f"Tabela removida: {destino}")
    except Exception as e:
        print(f"Erro ao remover tabela: {destino}. {e}")
    # Cria nova tabela (usando CSV atual para inferir tipos corretos)
    df = spark.read.option("header", True).csv(volume_path+arquivo)
    # Força IDs numéricos como long, evitando inferência string
    for col_id in ['id_cartao', 'id_conta', 'id_cliente', 'id_transacao', 'id_estorno']:
        if col_id in df.columns:
            df = df.withColumn(col_id, df[col_id].cast('long'))
    df.write.format('delta').mode('overwrite').option('mergeSchema', 'true').saveAsTable(destino)
    print(f"Tabela bronze criada: {destino} ({df.count()} registros)")